In [ ]:
import numpy as np
import cv2
print("NumPy version:", np.__version__)
print("OpenCV version:", cv2.__version__)

In [ ]:
%pip install "protobuf==3.20.3"

In [ ]:
%pip install "tensorflow==2.15.0" "scikit-learn" "matplotlib"

In [ ]:
import cv2
import numpy as mp
import matplotlib.pyplot as plt
import time
import os
import mediapipe as mp

## Keypoints using MP Holistic

In [ ]:
%pip install mediapipe==0.10.9

In [ ]:
## !pip install mediapipe opencv-python

In [ ]:
import mediapipe
print(mediapipe.__file__)

In [ ]:
import mediapipe as mp
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing model

In [ ]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Color conversion
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

In [ ]:
cv2.cvtColor??

In [ ]:
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION)
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    return image

In [ ]:
def draw_styled_landmarks(image, results):

    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION, 
                              mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                              mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
                             )
    ## landmark
    ## connection

    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(80,22,10), thickness=1, circle_radius=1),
                              mp_drawing.DrawingSpec(color=(80,44,121), thickness=1, circle_radius=1)
                             )
    
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(121,22,76), thickness=1, circle_radius=1),
                              mp_drawing.DrawingSpec(color=(121,44,250), thickness=1, circle_radius=1)
                             )
    
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(245,117,66), thickness=1, circle_radius=1),
                              mp_drawing.DrawingSpec(color=(245,66,230), thickness=1, circle_radius=1)
                             )


In [ ]:
mp_drawing.draw_landmarks

In [ ]:
cap = cv2.VideoCapture(0)                  ## Accessing camera
with mp_holistic.Holistic(min_detection_confidence = 0.5, min_tracking_confidence = 0.5) as holistic:
    while cap.isOpened():

        ## Read feed
        ret, frame = cap.read()   
    
        ## Make result
        image, results = mediapipe_detection(frame, holistic)
        #print(image)

        ## Draw landmarks
        draw_styled_landmarks(image, results)
        
        cv2.imshow("OpenCV Feed", image)       ## Show to screen
        if cv2.waitKey(10) & 0xFF == ord("q"): ## Breaking gracefully
            break
    cap.release()
    cv2.destroyAllWindows()

In [ ]:
frame

In [ ]:
draw_landmarks(frame, results)

In [ ]:
plt.imshow(cv2.cvtColor(draw_landmarks(frame, results), cv2.COLOR_BGR2RGB))

In [ ]:
results

## Extract Keypoints

In [ ]:
len(results.pose_landmarks.landmark)

In [ ]:
pose = []
for res in results.pose_landmarks.landmark:
    test = np.array([res.x, res.y, res.z, res.visibility])
    pose.append(test)

In [ ]:
pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(132)
face = np.array([[res.x, res.y, res.z, res.visibility] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(1404)
lh = np.array([[res.x, res.y, res.z, res.visibility] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
rh = np.array([[res.x, res.y, res.z, res.visibility] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)

In [ ]:
import numpy as np

def extract_keypoints(results):
    # Pose: 33 landmarks * 4 values (x, y, z, visibility) = 132
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    
    # Face: 468 landmarks * 3 values (x, y, z) = 1404 
    # Note: Face landmarks usually don't have visibility, so we use 3
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    
    # Left Hand: 21 landmarks * 3 values = 63
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    
    # Right Hand: 21 landmarks * 3 values = 63
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    
    return np.concatenate([pose, face, lh, rh])

In [ ]:
results_test = extract_keypoints(results)

In [ ]:
results_test

In [ ]:
np.save('0', results_test)

In [ ]:
np.load('0.npy')

## Setup Folders for Collection

In [ ]:
import os

# Path for exported data, numpy arrays
DATA_PATH = os.path.join("MP_Data")

# Actions that we try to detect
actions= np.array(['hello', 'thanks', 'iloveyou'])

# 30  videos
no_sequence = 30

# 30 Frames in 1 video
sequence_length = 30

In [ ]:
for action in actions:
    for sequence in range(no_sequence):
        try:
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except:
            pass

In [ ]:
# hello 
##1
##2

# thanks

# iloveyou

## 5. Collect KeypointValues for Training and Testing 

In [ ]:
cap = cv2.VideoCapture(0)                  ## Accessing camera
with mp_holistic.Holistic(min_detection_confidence = 0.5, min_tracking_confidence = 0.5) as holistic:

    # NEW loop
    #Loop through actions
    for action in actions:

        #Loop through sequences/videos
        for sequence in range(no_sequence):

            #Loop through video length/sequence length
            for frame_num in range(sequence_length):
            
                ## Read feed
                ret, frame = cap.read()   



                
                ## Make result
                image, results = mediapipe_detection(frame, holistic)
                print(image)
        
                ## Draw landmarks
                draw_styled_landmarks(image, results)

                #  NEW Apply wait logic
                if frame_num == 0: #If we are at frame 0
                    cv2.putText(image, 'STARTING COLLECTION', (120,200),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 4, cv2.LINE_AA)
                    cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    cv2.waitKey(2000)
                else:
                    cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)

                    cv2.imshow("OpenCV Feed", image)       ## Show to screen
                
                # NEW export keypoints
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)
            

                ## Breaking gracefully
                if cv2.waitKey(10) & 0xFF == ord("q"): 
                    break
    cap.release()
    cv2.destroyAllWindows()

## Preprocess Data and Create Labels and Features

In [ ]:
import tensorflow as tf
import mediapipe as mp
print("TensorFlow:", tf.__version__) # Should be 2.15.0
print("Mediapipe:", mp.__version__)   # Should be 0.10.9

In [ ]:
import tensorflow as tf
print(tf)

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [ ]:
label_map = {label:num for num, label in enumerate(actions)}

In [ ]:
label_map

In [ ]:
sequences, labels = [], []
for action in actions:
    for sequence in range(no_sequence):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

In [ ]:
np.array(sequences).shape

In [ ]:
# 90 videos, 30 frames, 1662 keypoints

In [ ]:
np.array(labels).shape

In [ ]:
X = np.array(sequences)
y = to_categorical(labels).astype(int)

In [ ]:
X

In [ ]:
y # 1,0,0 -> hello 0,1,0 -> thank you

## Build and Train LSTM Neural Network

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.05)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import TensorBoard

In [ ]:
log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir = log_dir)

In [ ]:
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30, 1662)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(actions.shape[0], activation='softmax'))

In [ ]:
# 64 LSTM Units

In [ ]:
model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])

In [ ]:
model.fit(X_train, y_train, epochs=300, callbacks=[tb_callback])

In [ ]:
print(X_train[0]) 

In [ ]:
model.summary()

## 8. Make Predictions

In [ ]:
res = model.predict(X_test)

In [ ]:
actions[np.argmax(res[0])]  # PREDICTED

In [ ]:
actions[np.argmax(y_test[0])] # ACTUAL

## 9. Save Weights

In [ ]:
model.save('action.h5')

In [ ]:
# model.load_weights('action.h5')

## 10. Evaluatinng using Confusion Matrix and Accuracy

In [ ]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [ ]:
y_hat = model.predict(X_train)

In [ ]:
y_true = np.argmax(y_train, axis=1).tolist()
y_hat = np.argmax(y_hat, axis=1).tolist()

In [ ]:
multilabel_confusion_matrix(y_true, y_hat)

In [ ]:
accuracy_score(y_true, y_hat)

## 11. Test in Real Time

In [ ]:
colors = [(245, 117, 16), (117, 245, 16), (16, 117, 245)]
def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        cv2.rectangle(output_frame, (0,60+num*40), (int(prob*100), 90+num*40), colors[num], -1)
        cv2.putText(output_frame, actions[num], (0, 85+num*40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)

    return output_frame


In [ ]:
plt.imshow(prob_viz(res, actions, image, colors))

In [ ]:
# New detection variable
sequence = []
sentence = [] # Concatinate
threshold = 0.7

cap = cv2.VideoCapture(0)                  ## Accessing camera
with mp_holistic.Holistic(min_detection_confidence = 0.5, min_tracking_confidence = 0.5) as holistic:
    while cap.isOpened():

        ## Read feed
        ret, frame = cap.read()   
    
        ## Make result
        image, results = mediapipe_detection(frame, holistic)
        print(image)


        # PREDICTING LOGIC
        keypoints = extract_keypoints(results)
        sequence.insert(0,keypoints)
        sequence = sequence[:30]

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            print(actions[np.argmax(res)])

            
            # VISUALIZATION LOGIC
            if res[np.argmax(res)]  > threshold:
                if len(sentence) > 0:
                    if actions[np.argmax(res)] != sentence[-1]:
                        sentence.append(actions[np.argmax(res)])
                else:
                    sentence.append(actions[np.argmax(res)])
    
            if len(sentence) > 5:
                sentence = sentence[-5:]

        cv2.rectangle(image, (0,0), (640,40), (245,117,16), -1)
        cv2.putText(image, ' '.join(sentence), (0,30),
                      cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        
        ## Draw landmarks
        draw_styled_landmarks(image, results)
        
        cv2.imshow("OpenCV Feed", image)       ## Show to screen
        if cv2.waitKey(10) & 0xFF == ord("q"): ## Breaking gracefully
            break
    cap.release()
    cv2.destroyAllWindows()

In [ ]:
cap.release()
cv2.destroyAllWindows()